### Notebook : 04_sql_agent


##### 1. Purpose

The SQL Agent is the first specialist execution agent in the multi-agent system.

Its responsibilities are to:

- Read the Coordinator execution plan.
- Determine whether a task is assigned to sql_agent.
- Skip execution when no SQL task exists.
- Call the existing SQL Analytics Tool when assigned.
- Convert the tool output into a safe Python structure.
- Validate the output using SQLAgentResult.
- Store the validated result in shared workflow state.
- Record execution history and errors.

The SQL Agent does not:

- Decide the overall workflow.
- Create execution plans.
- Perform churn prediction.
- Perform vector search.
- Recommend retention actions.
- Generate the final customer-facing response.


##### 2. Technologies Used

- Python
- Pydantic
- Typed shared state
- Dependency injection
- Databricks SQL
- PySpark row conversion
- Structured error handling


##### 3. Input

- The SQL Agent receives:

    ``` text

    MultiAgentState
        ├── user_request
        └── coordinator_result
                └── execution_plan

    ```

- It also receives the existing SQL Analytics Tool as a function.sql_analytics_tool(question: str)


##### 4. Output

- When execution succeeds, the SQL Agent produces a validated:

    ``` text

    SQLAgentResult
        ├── agent_name
        ├── status
        ├── message
        ├── task_description
        ├── error
        ├── sql_action
        └── sql_result

    ```
- The result is stored in: state["agent_results"]["sql_agent"]

- When no SQL task is assigned, the agent records a skipped execution and does not call the SQL tool.


##### 5. Architecture

``` text

Shared Workflow State
        │
        ▼
Read CoordinatorResult
        │
        ▼
Find sql_agent task
        │
        ├── No task
        │      │
        │      ▼
        │   Record skipped status
        │
        └── Task found
               │
               ▼
        Call SQL Analytics Tool
               │
               ▼
        Normalize tool output
               │
               ▼
        Validate SQLAgentResult
               │
               ▼
        Store in agent_results
               │
               ▼
        Record execution history

```


##### 6. Load Dependencies

In [0]:
%run ./01_shared_models


In [0]:
%run ./02_shared_state_and_helpers


##### 7. Additional Imports

In [0]:
from typing import Any, Dict, get_args
from pydantic import ValidationError
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole
w = WorkspaceClient()
LLM_MODEL = "databricks-meta-llama-3-1-8b-instruct"

##### 8. SQL Analytics Tool

###### 8.1 LLM Response Helper

In [0]:
#Sends a prompt to the Databricks serving endpoint.
#Returns the models text response.

def generate_answer(prompt: str) -> str:
    response = w.serving_endpoints.query(
        name=LLM_MODEL,
        messages=[
            ChatMessage(
                role=ChatMessageRole.USER,
                content=prompt
            )
        ],
        max_tokens=200,
        temperature=0.0
    )

    if (
        not response.choices
        or response.choices[0].message is None
        or not response.choices[0].message.content
    ):
        raise ValueError("The LLM returned no response.")

    return response.choices[0].message.content.strip()

###### 8.2 SQL Action Selection

In [0]:
#Builds the routing prompt.
#Uses the LLM to select one predefined SQL action.


def choose_sql_action(question):
    prompt = f"""
You are a SQL routing agent.

Choose the best SQL action for the user question.

Available actions:
1. count_churned_customers
2. churn_rate
3. count_month_to_month_customers

Question:
{question}

Return only one action name.
"""

    return generate_answer(prompt).strip().lower()    


###### 8.3 SQL Execution

In [0]:
#Maps the selected action to an approved SQL statement.
#Executes the SQL.
#Returns Spark Rows.

def run_sql_action(action):
    sql_map = {
        "count_churned_customers": """
            SELECT COUNT(*) AS churned_customers
            FROM dbw_agentic_ai_dev.telco_ai.gold_telco
            WHERE Churn_Flag = 1
        """,

        "churn_rate": """
            SELECT
                ROUND(100.0 * SUM(Churn_Flag) / COUNT(*), 2) AS churn_rate_percent
            FROM dbw_agentic_ai_dev.telco_ai.gold_telco
        """,

        "count_month_to_month_customers": """
            SELECT COUNT(*) AS month_to_month_customers
            FROM dbw_agentic_ai_dev.telco_ai.gold_telco
            WHERE Contract = 'Month-to-month'
        """
    }

    if action not in sql_map:
        return f"Unsupported SQL action: {action}"

    return spark.sql(sql_map[action]).collect()

###### 8.4 SQL Analytics Tool

In [0]:
def sql_analytics_tool(
    question: str,
) -> Dict[str, Any]:
    """
    Execute the existing SQL analytics workflow and return
    the dictionary contract expected by the SQL Agent.
    """

    try:
        sql_action = choose_sql_action(
            question
        )

        sql_result = run_sql_action(
            sql_action
        )

        return {
            "status": "success",
            "sql_action": sql_action,
            "sql_result": sql_result,
        }

    except Exception as exc:
        return {
            "status": "failed",
            "sql_action": "unknown",
            "sql_result": None,
            "error": str(exc),
        }


##### 9. Convert Tool Values into Safe Python Values

In [0]:
# The SQL Analytics Tool may return values such as: PySpark Row, Lists of rows, Dictionaries, Tuples, Primitive Python values
# Before storing the SQL result in shared state, we convert these values into standard Python structures.

def convert_to_python_value(
    value: Any,
) -> Any:
    """
    Recursively convert SQL and Spark values into
    standard Python structures.

    Parameters
    ----------
    value:
        Value returned by the SQL Analytics Tool.

    Returns
    -------
    Any
        JSON-compatible or standard Python value.
    """

    if value is None:
        return None

    if hasattr(value, "asDict"):
        return {
            key: convert_to_python_value(item)
            for key, item in value.asDict(
                recursive=True
            ).items()
        }

    if isinstance(value, dict):
        return {
            str(key): convert_to_python_value(item)
            for key, item in value.items()
        }

    if isinstance(value, list):
        return [
            convert_to_python_value(item)
            for item in value
        ]

    if isinstance(value, tuple):
        return [
            convert_to_python_value(item)
            for item in value
        ]

    if isinstance(
        value,
        (
            str,
            int,
            float,
            bool,
        ),
    ):
        return value

    return str(value)


##### 10. Validate the SQL Tool Response

In [0]:
# This validation checks the SQL tool contract before creating SQLAgentResult.

def validate_sql_tool_response(
    tool_response: Dict[str, Any],
) -> None:
    """
    Validate the basic contract returned by the
    SQL Analytics Tool.

    Parameters
    ----------
    tool_response:
        Dictionary returned by the SQL tool.

    Raises
    ------
    TypeError
        If the tool response is not a dictionary.

    ValueError
        If required fields are missing or the tool
        reports a failed status.
    """

    if not isinstance(tool_response, dict):
        raise TypeError(
            "SQL Analytics Tool must return a dictionary."
        )

    required_fields = {
        "status",
        "sql_action",
        "sql_result",
    }

    missing_fields = (
        required_fields - tool_response.keys()
    )

    if missing_fields:
        raise ValueError(
            "SQL Analytics Tool response is missing "
            f"required fields: {sorted(missing_fields)}"
        )

    if tool_response["status"] != "success":
        tool_error = tool_response.get(
            "error",
            "Unknown SQL tool error.",
        )

        raise ValueError(
            "SQL Analytics Tool failed: "
            f"{tool_error}"
        )

    sql_action = tool_response["sql_action"]

    if not isinstance(sql_action, str):
        raise TypeError(
            "SQL tool failed 'sql_action' must be "
            "a string."
        )

    if not sql_action.strip():
        raise ValueError(
            "SQL tool failed 'sql_action' cannot "
            "be empty."
        )


##### 11. Execute SQL Agent Logic

In [0]:
def execute_sql_agent(
    user_request: str,
    task: AgentTask,
    sql_analytics_tool: SQLAnalyticsFunction,
) -> SQLAgentResult:
    """
    Execute the SQL task assigned by the Coordinator.

    Parameters
    ----------
    user_request:
        Original customer analytics request.

    task:
        Coordinator task assigned to sql_agent.

    sql_analytics_tool:
        Existing SQL Analytics Tool function.

    Returns
    -------
    SQLAgentResult
        Validated structured SQL Agent result.

    Raises
    ------
    ValueError
        If the request or task is invalid.

    TypeError
        If the SQL tool response has an invalid type.

    ValidationError
        If the final result violates SQLAgentResult.
    """

    cleaned_request = user_request.strip()

    if not cleaned_request:
        raise ValueError(
            "SQL Agent requires a non-empty "
            "customer request."
        )

    if task.agent_name != SQL_AGENT_NAME:
        raise ValueError(
            "The assigned task does not belong to "
            "sql_agent."
        )

    tool_response = sql_analytics_tool(
        cleaned_request
    )

    validate_sql_tool_response(
        tool_response=tool_response
    )

    normalized_sql_result = (
        convert_to_python_value(
            tool_response["sql_result"]
        )
    )

    result_payload = {
        "agent_name": SQL_AGENT_NAME,
        "status": "success",
        "message": (
            "SQL analytics task completed "
            "successfully."
        ),
        "task_description": (
            task.task_description
        ),
        "error": None,
        "sql_action": (
            tool_response["sql_action"]
        ),
        "sql_result": normalized_sql_result,
    }

    sql_agent_result = (
        SQLAgentResult.model_validate(
            result_payload
        )
    )

    return sql_agent_result


##### 12. Shared-State Runner

In [0]:
def run_sql_agent(
    state: MultiAgentState,
    sql_analytics_tool: SQLAnalyticsFunction,
) -> MultiAgentState:
    """
    Run the SQL Agent inside the shared workflow.

    The function:

    1. Reads the Coordinator result.
    2. Finds the task assigned to sql_agent.
    3. Skips when no SQL task is assigned.
    4. Validates task dependencies.
    5. Calls the SQL Analytics Tool.
    6. Stores the validated SQLAgentResult.
    7. Records execution history.
    8. Records errors when execution fails.

    Parameters
    ----------
    state:
        Current multi-agent shared state.

    sql_analytics_tool:
        Existing SQL Analytics Tool function.

    Returns
    -------
    MultiAgentState
        Updated shared workflow state.
    """

    try:
        coordinator_result = state.get(
            "coordinator_result"
        )

        if coordinator_result is None:
            raise ValueError(
                "SQL Agent cannot run because "
                "coordinator_result is missing."
            )

        assigned_task = find_assigned_agent_task(
            coordinator_result=coordinator_result,
            agent_name=SQL_AGENT_NAME,
        )

        if assigned_task is None:
            record_agent_execution(
                state=state,
                agent_name=SQL_AGENT_NAME,
                status="skipped",
                message=(
                    "SQL Agent was not required by "
                    "the execution plan."
                ),
            )

            return state

        validate_task_dependencies(
            state=state,
            task=assigned_task,
        )

        sql_agent_result = execute_sql_agent(
            user_request=state["user_request"],
            task=assigned_task,
            sql_analytics_tool=(
                sql_analytics_tool
            ),
        )

        store_agent_result(
            state=state,
            agent_result=sql_agent_result,
        )

        record_agent_execution(
            state=state,
            agent_name=SQL_AGENT_NAME,
            status="success",
            message="SQL task completed successfully.",
        )

    except (
        ValueError,
        ValidationError,
        TypeError,
        KeyError,
    ) as exc:
        error_message = str(exc)

        record_agent_execution(
            state=state,
            agent_name=SQL_AGENT_NAME,
            status="failed",
            message=(
                "SQL Agent failed to complete "
                "the assigned task."
            ),
        )

        record_agent_error(
            state=state,
            agent_name=SQL_AGENT_NAME,
            error_code="SQL_AGENT_ERROR",
            error_message=error_message,
        )

    except Exception as exc:
        error_message = (
            "Unexpected SQL Agent error: "
            f"{exc}"
        )

        record_agent_execution(
            state=state,
            agent_name=SQL_AGENT_NAME,
            status="failed",
            message=(
                "SQL Agent encountered an "
                "unexpected error."
            ),
        )

        record_agent_error(
            state=state,
            agent_name=SQL_AGENT_NAME,
            error_code=(
                "SQL_AGENT_UNEXPECTED_ERROR"
            ),
            error_message=error_message,
        )

    return state


##### 13. Independent Test Function

In [0]:
# The verification function uses a mock SQL tool. It does not call a real SQL warehouse or table.

def test_sql_agent_unit_test() -> None:
    """
    Run deterministic unit-style tests for the SQL Agent.

    Tests:

    1. Successful SQL execution.
    2. Clean skip when no SQL task is assigned.
    3. Error handling for an invalid SQL tool result.
    """
  

    ALLOWED_REQUEST_TYPES = list(
        get_args(RequestType)
    )

    sql_task = AgentTask(
        task_id="task_1",
        agent_name=SQL_AGENT_NAME,
        task_description=(
            "Run the assigned SQL analytics task."
        ),
        depends_on=[],
    )

    final_response_task = AgentTask(
        task_id="task_2",
        agent_name="final_response_agent",
        task_description=(
            "Generate the final grounded response."
        ),
        depends_on=["sql_agent"],
    )    

    coordinator_result = CoordinatorResult(
        agent_name=COORDINATOR_AGENT_NAME,
        status="success",
        message="Execution plan created successfully.",
        task_description=None,
        error=None,
        request_type="sql_analytics",
        reasoning=(
            "The request requires SQL analytics, so the SQL Agent "
            "should execute first and the Final Response Agent should "
            "use its result."
        ),
        execution_plan=[
            sql_task,
            final_response_task,
        ],
)

    def mock_successful_sql_tool(
        question: str,
    ) -> Dict[str, Any]:
        assert isinstance(question, str)
        assert question.strip()

        return {
            "tool": "sql_analytics_tool",
            "status": "success",
            "sql_action": "test_sql_action",
            "sql_result": [
                {
                    "result_value": 100
                }
            ],
        }

    success_state = create_initial_state(
        "Run a SQL analytics test."
    )

    success_state["coordinator_result"] = (
        coordinator_result
    )

    updated_success_state = run_sql_agent(
        state=success_state,
        sql_analytics_tool=(
            mock_successful_sql_tool
        ),
    )

    assert (
        SQL_AGENT_NAME
        in updated_success_state["agent_results"]
    )

    stored_result = updated_success_state[
        "agent_results"
    ]["sql_agent"]

    assert isinstance(
        stored_result,
        SQLAgentResult,
    )

    assert stored_result.status == "success"

    assert (
        stored_result.sql_action
        == "test_sql_action"
    )

    assert stored_result.sql_result == [
        {
            "result_value": 100
        }
    ]

    assert (
        updated_success_state[
            "execution_history"
        ][0].status
        == "success"
    )

    assert updated_success_state["errors"] == []

    skip_coordinator_result = CoordinatorResult(
        agent_name=COORDINATOR_AGENT_NAME,
        status="success",
        message="Execution plan created successfully.",
        task_description=None,
        error=None,
        request_type="sql_analytics",
        reasoning=(
            "Only the final response agent is included "
            "to test SQL Agent skip behavior."
        ),
        execution_plan=[
            AgentTask(
                task_id="task_1",
                agent_name="final_response_agent",
                task_description=(
                    "Generate the final response."
                ),
                depends_on=[],
            )
    ],
)

    skip_state = create_initial_state(
        "Run a non-SQL workflow test."
    )

    skip_state["coordinator_result"] = (
        skip_coordinator_result
    )

    updated_skip_state = run_sql_agent(
        state=skip_state,
        sql_analytics_tool=(
            mock_successful_sql_tool
        ),
    )

    assert (
        SQL_AGENT_NAME
        not in updated_skip_state["agent_results"]
    )

    assert (
        updated_skip_state[
            "execution_history"
        ][0].status
        == "skipped"
    )

    assert updated_skip_state["errors"] == []

    def mock_invalid_sql_tool(
        question: str,
    ) -> Dict[str, Any]:
        return {
            "status": "success",
            "sql_action": "invalid_test",
        }

    failure_state = create_initial_state(
        "Run a SQL failure test."
    )

    failure_state["coordinator_result"] = (
        coordinator_result
    )

    updated_failure_state = run_sql_agent(
        state=failure_state,
        sql_analytics_tool=(
            mock_invalid_sql_tool
        ),
    )

    assert (
        SQL_AGENT_NAME
        not in updated_failure_state[
            "agent_results"
        ]
    )

    assert (
        updated_failure_state[
            "execution_history"
        ][0].status
        == "failed"
    )

    assert len(
        updated_failure_state["errors"]
    ) == 1

    assert (
        updated_failure_state[
            "errors"
        ][0].error_code
        == "SQL_AGENT_ERROR"
    )

    print(
        "All SQL Agent tests passed."
    )


##### 14. SQL Agent Integration Test

In [0]:
def test_sql_agent_integration_test() -> None:
    """
    Test SQL Agent with the real SQL Analytics Tool.
    """

    sql_task = AgentTask(
        task_id="task_1",
        agent_name=SQL_AGENT_NAME,
        task_description=(
            "Count churned customers."
        ),
        depends_on=[],
    )

    final_task = AgentTask(
        task_id="task_2",
        agent_name=FINAL_RESPONSE_AGENT_NAME,
        task_description=(
            "Generate the final response."
        ),
        depends_on=[
            SQL_AGENT_NAME,
        ],
    )

    coordinator_result = CoordinatorResult(
        status="success",
        message=(
            "Execution plan created successfully."
        ),
        request_type="sql_analytics",
        reasoning=(
            "The request requires SQL analytics."
        ),
        execution_plan=[
            sql_task,
            final_task,
        ],
    )

    state = create_initial_state(
        "How many customers churned?"
    )

    state["coordinator_result"] = (
        coordinator_result
    )

    updated_state = run_sql_agent(
        state=state,
        sql_analytics_tool=sql_analytics_tool,
    )

    print("SQL Agent Result:")
    print(
        updated_state[
            "agent_results"
        ].get(SQL_AGENT_NAME)
    )

    print("\nExecution History:")
    print(
        updated_state["execution_history"]
    )

    print("\nErrors:")
    print(updated_state["errors"])

    assert (
    SQL_AGENT_NAME
    in updated_state["agent_results"]
    )

    result = updated_state[
        "agent_results"
    ][SQL_AGENT_NAME]

    assert result.status == "success"

    assert (
        result.sql_action
        == "count_churned_customers"
    )

    assert result.sql_result == [
        {
            "churned_customers": 1869
        }
    ]

    assert len(
        updated_state["execution_history"]
    ) == 1

    assert (
        updated_state[
            "execution_history"
        ][0].status
        == "success"
    )

    assert updated_state["errors"] == []

##### 15. Understanding the Skip Behavior

- The Coordinator may decide that SQL analytics are not required.

- In that case: ``` assigned_task is None ```

- The SQL Agent:

    - Does not call the SQL tool.
    - Does not create SQLAgentResult.
    - Does not add an entry to agent_results.
    - Records that its execution was skipped.
    - Returns the state without failing the workflow.

- This is an intentional workflow outcome, not an error.


##### 16. Key Learnings

- The Coordinator plans the work; the SQL Agent executes only its assigned task.

- Specialist agents should not make workflow-routing decisions.

- An agent may be skipped when it has no assigned task.

- Skipping is a normal workflow outcome rather than an error.

- Dependency injection prevents the agent from being tightly coupled to a specific tool implementation.

- Tool responses should be validated before they are trusted.

- PySpark Row values should be converted into standard Python structures.

- Pydantic validates the final SQL Agent result contract.

- Shared state stores validated agent objects rather than unstructured values.

- Execution history records success, failure, and skipped outcomes.

- Expected errors and unexpected errors are classified separately.

- Core execution logic is separated from shared-state management.


##### 17. Conclusion

In this notebook, first specialist execution agent was implemented.

The SQL Agent:

- Reads the Coordinator execution plan.
- Finds its assigned task.
- Skips safely when no SQL task exists.
- Validates dependencies.
- Calls the SQL Analytics Tool.
- Validates the tool response.
- Converts SQL and Spark values into standard Python structures.
- Creates a validated SQLAgentResult.
- Stores the result in shared state.
- Records execution history and errors.

The key separation is:

execute_sql_agent() --->   Core SQL Agent execution logic

run_sql_agent() --->     Shared-state and workflow integration


##### 18. Next Notebook

The next notebook is: 05_prediction_agent

The Prediction Agent will:

- Read the Coordinator execution plan.
- Determine whether prediction is required.
- Extract the customer ID when needed.
- Call the existing churn prediction tool.
- Handle model probabilities and confidence.
- Validate the result using PredictionAgentResult.
- Store the result in shared state.
- Skip cleanly when no prediction task is assigned.
- Record execution history and errors.